In [74]:
import numpy as np 
import soundfile as sf
import librosa
import pandas as pd 
from pathlib import Path
import pickle 
import IPython.display as ipd
import sys 
sys.path.append('../../datasets/spatial_audio_pipeline/spatial_audio_util/')
import util_audio 
import soxr

# Make stimuli for 2024 mono word recognition experiment

## Wanted Conditions:
### SNRs:
-9, -6, -3, 0, 3, inf 
### Distractor conditions:
- English (sex balanced: 2 same, 2 different per SNR )
- Mandarin (sex balanced: 2 same, 2 different per SNR )
- Dutch (sex balanced: 2 same, 2 different per SNR )
- 2-distractor
- 4-distractor
- 8-talker babble 
- Speech Shaped Noise
- Natural scenes 
- Music


Will be using cues, foregrounds, and english distractors for 1, 2, and 4 distractor cases taken from manifest:
`/om/user/imgriff/datasets/human_word_rec_SWC_2024/full_cue_target_distractor_df_w_meta_transcripts.pdpkl`
* for 4 distractor case, combine sets of 2 distractors (1 male set 1 female set = 2 of each = 4 total)

This manifest and
stimuli will be moved to `/om/user/imgriff/datasets/human_word_rec_SWC_2024`


##### NOTE: Sounds actually cut and written using `src/get_swc_mono_expmnt_stim_2024`. This notebook prepares the manifest for that script.

In [109]:
stim_out_path = Path('/om/user/imgriff/datasets/human_word_rec_SWC_2024')
manifest = pd.read_pickle('/om/user/imgriff/datasets/human_word_rec_SWC_2024/full_cue_target_distractor_df_w_meta_transcripts.pdpkl')
manifest.shape

(1952, 53)

In [116]:
bad_eg_df = manifest[(manifest.word == 'album') & (manifest.gender == 'female')]
bad_eg_df

,manifest_ix,orig_df_ix,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,gender,gender_int,split,...,excerpt_src_fn,excerpt_cue_src_fn,excerpt_distractor_1_src_fn,excerpt_distractor_2_src_fn,src_fn_y,distractor_1_src_fn,distractor_2_src_fn,target_transcripts,distractor_1_transcripts,distractor_2_transcripts
12,12,537209,bowie-media,0.58,91.45,90.87,swc,female,0,NaN,...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,"[soundtrack, album]","[a, disabled, person, is, dissatisfied]","[one thousand, nine hundred and ninety, seven,..."
988,988,537209,bowie-media,0.58,91.45,90.87,swc,female,0,NaN,...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,"[soundtrack, album]","[navigates, the, website]","[led, them, to, lose, their, early, advantage]"


In [123]:
ix = 1 
ipd.display(ipd.Audio(bad_eg_df['excerpt_cue_src_fn'].iloc[ix]))
ipd.display(ipd.Audio(bad_eg_df['excerpt_src_fn'].iloc[ix]))

In [76]:
# Make condition dict 
import itertools 
import pickle 
import numpy as np 
snrs = np.arange(-9, 4, 3).tolist() # -9, -6, -3, 0, 3

# conditions as above
conditions = ['background_musdb18hq',
              'background_cv08talkerbabble',
              'background_issnstationary',
              'background_ieeeaaspcasa',
              "1-talker-english-same",
              "1-talker-english-different",
              "1-talker-mandarin-same",
              "1-talker-mandarin-different",
              "1-talker-dutch-same",
              "1-talker-dutch-different",
              "2-talker",
              "4-talker"]

condition_pairs = list(itertools.product(conditions, snrs))
condition_pairs.append(('SILENCE', 'inf'))
cond_ix_dict = {ix:cond for ix, cond in enumerate(condition_pairs)}

# out_name = stim_out_path/"human_attn_expmt_cond_map.pkl"
# write condition dict as pickle 
# with open(out_name, 'wb') as f:
#     pickle.dump(cond_ix_dict, f)


In [77]:
[k for k, v in cond_ix_dict.items() if "1-talker" in v[0] ]

[20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49]

In [78]:
len(cond_ix_dict)

61

In [108]:
# Make word key 
import json
target_words = np.sort(manifest['word'].unique())
#enumerate words as dict 
target_word_dict = dict(enumerate(target_words))
# save as pickle
out_name = stim_out_path / "human_attn_expmt_word_key.pkl"
# with open(out_name, 'wb') as f:
    # pickle.dump(target_word_dict, f)

words = list(target_word_dict.values())
# save as json 
out_name = stim_out_path /  "human_attn_expmt_word_key.js"
with open(out_name, 'w') as f:
    json.dump({"dictionary":words}, f)



Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



In [81]:
target_word_dict

{0: 'about',
 1: 'above',
 2: 'access',
 3: 'across',
 4: 'action',
 5: 'active',
 6: 'added',
 7: 'africa',
 8: 'after',
 9: 'again',
 10: 'against',
 11: 'airport',
 12: 'album',
 13: 'allowed',
 14: 'almost',
 15: 'alone',
 16: 'along',
 17: 'already',
 18: 'always',
 19: 'america',
 20: 'among',
 21: 'ancient',
 22: 'another',
 23: 'appear',
 24: 'around',
 25: 'artist',
 26: 'asked',
 27: 'attack',
 28: 'author',
 29: 'award',
 30: 'based',
 31: 'battle',
 32: 'beach',
 33: 'became',
 34: 'because',
 35: 'become',
 36: 'before',
 37: 'began',
 38: 'behind',
 39: 'being',
 40: 'believe',
 41: 'below',
 42: 'better',
 43: 'between',
 44: 'black',
 45: 'board',
 46: 'border',
 47: 'bridge',
 48: 'bright',
 49: 'bring',
 50: 'british',
 51: 'brother',
 52: 'brought',
 53: 'brown',
 54: 'built',
 55: 'buried',
 56: 'called',
 57: 'canada',
 58: 'cannot',
 59: 'capital',
 60: 'career',
 61: 'carried',
 62: 'cause',
 63: 'center',
 64: 'central',
 65: 'certain',
 66: 'change',
 67: 'char

### Flatten same vs diff distractor sex rows into columns so cues are same across experiment

In [83]:
### For four distractor condition - pivot manifest to remove sex_cond column, and add values in distractor_1 distractor_2 columns for those rows as new columns distractor_3 distractor_4 
same_manifest = manifest[manifest.sex_cond == 'same'].copy()
diff_manifest = manifest[manifest.sex_cond == 'different'].copy()

# add _same_ to column names with _distractor_ in them for same_manifest 
new_cols = [col.replace('distractor_1', 'same_sex_distractor_1').replace('distractor_2', 'same_sex_distractor_2') for col in same_manifest.columns]
same_manifest.columns = new_cols
same_manifest['same_sex_dist_1_word'], same_manifest['same_sex_dist_2_word'] = zip(*same_manifest.apply(lambda x: (x.distractor_word[0], x.distractor_word[1]), axis=1))
## add different_sex to diff column names 
new_cols = [col.replace('distractor_1', 'diff_sex_distractor_1').replace('distractor_2', 'diff_sex_distractor_2') for col in diff_manifest.columns]
diff_manifest.columns = new_cols
diff_manifest['diff_sex_dist_1_word'], diff_manifest['diff_sex_dist_2_word'] = zip(*diff_manifest.apply(lambda x: (x.distractor_word[0], x.distractor_word[1]), axis=1))

# merge same_manifest and diff_manifest on client_id and word
on_cols = ['client_id', 'word']
human_df = pd.merge(same_manifest, diff_manifest, left_on=on_cols, right_on=on_cols, how='left')
human_df = human_df[[col for col in human_df.columns if "_y" not in col]].copy()
# remove _x from column names 
new_cols = [col.replace('_x', '') for col in human_df.columns]
human_df.columns = new_cols


In [84]:
col_to_drop = ['total_file_duration_in_s',
       'distractor_orig_df_ix', 'distractor_client_id',
       'distractor_clip_dur_in_s', 'distractor_clip_end_in_s',
       'distractor_clip_start_in_s', 'distractor_corpus', 'distractor_gender',
       'distractor_gender_int', 'distractor_split', 'distractor_split_int',
       'distractor_sr', 'distractor_src_fn',
       'distractor_total_file_duration_in_s', 'distractor_word',
       'clip_dur_in_s', 'split', 'split_int',
       'clip_end_in_s', 'clip_start_in_s', 
       'cue_clip_dur_in_s',
       'cue_clip_end_in_s', 'cue_clip_start_in_s',
        'cue_split', 'cue_split_int',  'cue_src_fn',
         'src_fn',
       ]

human_df = human_df.drop(columns=col_to_drop)

In [85]:
human_df.columns

Index(['manifest_ix', 'orig_df_ix', 'client_id', 'corpus', 'gender',
       'gender_int', 'sr', 'word', 'cue_client_id', 'cue_corpus', 'cue_gender',
       'cue_gender_int', 'cue_sr', 'cue_total_file_duration_in_s', 'cue_word',
       'sex_cond', 'excerpt_src_fn', 'excerpt_cue_src_fn',
       'excerpt_same_sex_distractor_1_src_fn',
       'excerpt_same_sex_distractor_2_src_fn', 'same_sex_distractor_1_src_fn',
       'same_sex_distractor_2_src_fn', 'target_transcripts',
       'same_sex_distractor_1_transcripts',
       'same_sex_distractor_2_transcripts', 'same_sex_dist_1_word',
       'same_sex_dist_2_word', 'excerpt_diff_sex_distractor_1_src_fn',
       'excerpt_diff_sex_distractor_2_src_fn', 'diff_sex_distractor_1_src_fn',
       'diff_sex_distractor_2_src_fn', 'diff_sex_distractor_1_transcripts',
       'diff_sex_distractor_2_transcripts', 'diff_sex_dist_1_word',
       'diff_sex_dist_2_word'],
      dtype='object')

In [106]:
print(f"Exampes per gender \n{human_df['gender'].value_counts()}")
print(f"All words occur twice? = {all(human_df.word.value_counts() == 2)}")
print(f"N unique words: {human_df.word.nunique()}")
print(f"All unique words in target word dict? = {(set(human_df.word.unique()) == set(target_word_dict.values()))}")

Exampes per gender 
female    488
male      488
Name: gender, dtype: int64
All words occur twice? = True
N unique words: 488
All unique words in target word dict? = True


In [11]:
# sort manifest by word 
human_df = human_df.sort_values('word').reset_index(drop=True)

## Add rest of distractor paths as columns and save 


###  Add foreign language distractors 

In [12]:
n_per_gender = human_df.gender.value_counts()[0] # is same for 0 or 1 ix 
cv_stim_dir = Path("/om2/data/public/mozilla-CommonVoice-9.0/cv-corpus-9.0-2022-04-27") 


#### Load and cut Mandarin subset


In [13]:
zh_df = pd.read_csv('/om2/data/public/mozilla-CommonVoice-9.0/cv-corpus-9.0-2022-04-27/zh-CN/dev.tsv', sep="\t")
zh_df = zh_df[zh_df.gender.isin(['male', 'female'])]
print(zh_df.gender.value_counts())
zh_df = zh_df.groupby('gender').sample(n_per_gender, replace=False, random_state=0).reset_index(drop=True)
print(zh_df.gender.value_counts())


male      4373
female    1030
Name: gender, dtype: int64
female    488
male      488
Name: gender, dtype: int64


In [14]:
### Add full file path to dir to zh_df
zh_stim_dir = cv_stim_dir / "zh-CN/clips" 

zh_df['full_path'] = zh_stim_dir / zh_df.path

## Add full path to human_df
human_df.loc[human_df.gender == 'male', 'same_sex_zh_distractor_full_path'] = zh_df.loc[zh_df.gender == 'male', 'full_path'].values
human_df.loc[human_df.gender == 'female', 'same_sex_zh_distractor_full_path'] = zh_df.loc[zh_df.gender == 'female', 'full_path'].values

human_df.loc[human_df.gender == 'male', 'diff_sex_zh_distractor_full_path'] = zh_df.loc[zh_df.gender == 'female', 'full_path'].values
human_df.loc[human_df.gender == 'female', 'diff_sex_zh_distractor_full_path'] = zh_df.loc[zh_df.gender == 'male', 'full_path'].values

#### Load and cut Dutch subset


In [15]:
nl_df = pd.read_csv('/om2/data/public/mozilla-CommonVoice-9.0/cv-corpus-9.0-2022-04-27/nl/dev.tsv', sep="\t")
nl_df = nl_df[nl_df.gender.isin(['male', 'female'])]
print(nl_df.gender.value_counts())
nl_df = nl_df.groupby('gender').sample(n_per_gender, replace=False, random_state=0).reset_index(drop=True)
print(nl_df.gender.value_counts())

nl_stim_dir = cv_stim_dir / "nl/clips"

nl_df['full_path'] = nl_stim_dir / nl_df.path

## Add full path to human_df
human_df.loc[human_df.gender == 'male', 'same_sex_nl_distractor_full_path'] = nl_df.loc[nl_df.gender == 'male', 'full_path'].values
human_df.loc[human_df.gender == 'female', 'same_sex_nl_distractor_full_path'] = nl_df.loc[nl_df.gender == 'female', 'full_path'].values

human_df.loc[human_df.gender == 'male', 'diff_sex_nl_distractor_full_path'] = nl_df.loc[nl_df.gender == 'female', 'full_path'].values
human_df.loc[human_df.gender == 'female', 'diff_sex_nl_distractor_full_path'] = nl_df.loc[nl_df.gender == 'male', 'full_path'].values

male      5282
female    1786
Name: gender, dtype: int64
female    488
male      488
Name: gender, dtype: int64


## Add noise condition file paths to manifest

In [16]:
path_to_bg_stim = Path("/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/")
# get conditions we actually need to source - music, babble, natural noise. will make ssn on the fly 
conditions = ['background_musdb18hq',
              'background_cv08talkerbabble',
              'background_ieeeaaspcasa',
            ]

n_to_sample = human_df.shape[0]

In [17]:
np.random.seed(0)
for cond in conditions:
    list_of_wavs = list((path_to_bg_stim / cond).glob(f"*.wav"))
    sample = np.random.choice(list_of_wavs, size=n_to_sample, replace=True)
    cond_str = 'music' if 'mus' in cond else 'babble' if 'babble' in cond else 'natural_scene'
    human_df[f"{cond_str}_full_path"] = sample

In [18]:
human_df.head(2)

,manifest_ix,orig_df_ix,client_id,corpus,gender,gender_int,sr,word,cue_client_id,cue_corpus,...,diff_sex_distractor_2_transcripts,diff_sex_dist_1_word,diff_sex_dist_2_word,same_sex_zh_distractor_full_path,diff_sex_zh_distractor_full_path,same_sex_nl_distractor_full_path,diff_sex_nl_distractor_full_path,music_full_path,babble_full_path,natural_scene_full_path
0,0,601538,laura-s,swc,female,0,44100,about,laura-s,swc,...,"[and, release, of, final, fantasy, vi, by]",seven,final,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...
1,488,236668,tonyle,swc,male,1,44100,about,tonyle,swc,...,[design],however,design,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...


In [26]:
row = human_df.loc[0]

In [28]:
row.gender[0]

'f'

In [20]:
### Write out human stim manifest ###
out_name = stim_out_path / "human_cue_target_distractor_df_w_meta_transcripts.pdpkl"
human_df.to_pickle(out_name)

In [55]:
x = [np.arange(3), np.arange(4,7)]

In [58]:
np.sum(x, axis=0)

array([4, 6, 8])

In [52]:
import sys 
sys.path.append('../../datasets/spatial_audio_pipeline/spatial_audio_util/')
import util_audio 

### After sounds cut in .py script, add back f0 for 1-talker distractor case

In [34]:
import functools as ft


In [65]:
stim_out_path = Path('/om/user/imgriff/datasets/human_word_rec_SWC_2024/')
manifest_df = pd.read_pickle( stim_out_path/ "human_cue_target_distractor_df_w_meta_transcripts.pdpkl") 

all_dfs = list(stim_out_path.glob("*1_distractor_f0_manifest.pdpkl"))
all_f0_meta_dfs = [pd.read_pickle(f) for f in all_dfs] 

## convert floats to float32 for each column with f0 in the name to prevent merge failure due to float precision
for df in all_f0_meta_dfs:
    f0_cols = [col for col in df.columns if 'f0' in col]
    for col in f0_cols:
        df[col] = df[col].astype(np.float32)

## merge manifests 
f0_meta_df = ft.reduce(lambda left, right: pd.merge(left, right, on=['df_ixs', 'target_f0']), all_f0_meta_dfs)
assert f0_meta_df.shape[0] == manifest_df.shape[0], "Lost rows due to merge failue - check f0 rounding!"
f0_meta_df.head()

,df_ixs,target_f0,mandarin_distractor_f0,dutch_distractor_f0,english_distractor_f0
0,0,190.620972,219.545975,160.770721,229.817505
1,1,105.636078,155.406250,110.087585,114.206657
2,2,155.351135,237.625198,222.443344,193.325760
3,3,127.755608,99.292030,153.111374,148.092407
4,4,236.283768,227.638489,181.170914,194.650116


#### Add f0 columns to manifest

In [69]:
cols_to_add = ['target_f0', "english_distractor_f0", "mandarin_distractor_f0", "dutch_distractor_f0"]
manifest_df.loc[manifest_df.index, cols_to_add] = f0_meta_df.loc[f0_meta_df.index, cols_to_add].values
manifest_df.head()

,manifest_ix,orig_df_ix,client_id,corpus,gender,gender_int,sr,word,cue_client_id,cue_corpus,...,diff_sex_zh_distractor_full_path,same_sex_nl_distractor_full_path,diff_sex_nl_distractor_full_path,music_full_path,babble_full_path,natural_scene_full_path,target_f0,english_distractor_f0,mandarin_distractor_f0,dutch_distractor_f0
0,0,601538,laura-s,swc,female,0,44100,about,laura-s,swc,...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,190.620972,229.817505,219.545975,160.770721
1,488,236668,tonyle,swc,male,1,44100,about,tonyle,swc,...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,105.636078,114.206657,155.406250,110.087585
2,1,638828,dolliellama,swc,female,0,44100,above,dolliellama,swc,...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,155.351135,193.325760,237.625198,222.443344
3,489,252228,alexkillby,swc,male,1,44100,above,alexkillby,swc,...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,127.755608,148.092407,99.292030,153.111374
4,2,249863,ama1016,swc,female,0,44100,access,ama1016,swc,...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,236.283768,194.650116,227.638489,181.170914


In [72]:
stim_out_path

PosixPath('/om/user/imgriff/datasets/human_word_rec_SWC_2024')

In [70]:
### Write out new df with f0s 

manifest_df.to_pickle( stim_out_path/ "human_cue_target_distractor_df_w_meta_transcripts_w_f0.pdpkl") 


In [71]:
manifest_df

,manifest_ix,orig_df_ix,client_id,corpus,gender,gender_int,sr,word,cue_client_id,cue_corpus,...,diff_sex_zh_distractor_full_path,same_sex_nl_distractor_full_path,diff_sex_nl_distractor_full_path,music_full_path,babble_full_path,natural_scene_full_path,target_f0,english_distractor_f0,mandarin_distractor_f0,dutch_distractor_f0
0,0,601538,laura-s,swc,female,0,44100,about,laura-s,swc,...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,190.620972,229.817505,219.545975,160.770721
1,488,236668,tonyle,swc,male,1,44100,about,tonyle,swc,...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,105.636078,114.206657,155.406250,110.087585
2,1,638828,dolliellama,swc,female,0,44100,above,dolliellama,swc,...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,155.351135,193.325760,237.625198,222.443344
3,489,252228,alexkillby,swc,male,1,44100,above,alexkillby,swc,...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,127.755608,148.092407,99.292030,153.111374
4,2,249863,ama1016,swc,female,0,44100,access,ama1016,swc,...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,236.283768,194.650116,227.638489,181.170914
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
971,485,536345,bowie-media,swc,female,0,48000,wrote,bowie-media,swc,...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,187.901505,190.115372,151.157425,161.634552
972,974,479509,messedrocker,swc,male,1,44100,yellow,messedrocker,swc,...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,231.521500,115.157524,123.922554,141.201233
973,486,113916,popularoutcast,swc,female,0,44100,yellow,popularoutcast,swc,...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/spatial_audio_pipeline/asse...,171.579788,189.391144,187.242538,248.804810
974,487,135601,popularoutcast,swc,female,0,44100,young,popularoutcast,swc,...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/data/public/mozilla-CommonVoice-9.0/cv-co...,/om2/user/msaddler/spatial_audio_pipeline/asse...,/om2/user/msaddler/